- [] Focus at 1 horizon e.g 1 = short and segmenting (ID grouped) rows of features by quantiles relative to the available horizons
- [] Add Lag features upon focusing on the first
- [] Correlation against the code/sub_code/sub_categories .. if plausible to set these as targets or whether an independent model should be used for this

# Config

In [1]:
from pathlib import Path 

import numpy as np
import pandas as pd 
import polars as pl

# /kaggle/input/ts-prediction/train.parquet
# /kaggle/input/ts-prediction/test.parquet

DATA_DIR = Path("/kaggle/input/ts-prediction")

assert DATA_DIR.exists() and DATA_DIR.is_dir(), "Could not find input directory"

TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"

assert TRAIN_PATH.exists() and TEST_PATH.exists(), "Expected both train and test file paths to exist but at least one is missing from mounted kernel."

SAMPLE_SIZE = 50_000

# Utils

In [2]:
# Evaluation Functions
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

def eval(y_target, y_pred, w):
    score = weighted_rmse_score(y_target, y_pred, w)
    residue = y_target - y_pred 
    try:
        return {
            "weighted_rmse_score": score, 
            "residue": residue,
            "rmse": np.sqrt(np.mean(residue**2)).round(4).item(),
            "bias^2": (residue.mean()**2).round(4).item(),
            "var": residue.var().round(4).item(),
            "wrmse_rmse": np.sqrt(np.sum(w * residue**2)/np.sum(w)).round(4).item()
        }
    except Exception as e: 
        print(f"Error captured: {e}")
        return {
            "weighted_rmse_score": score,
            "residue": y_target - y_pred
        }

# Prep Features

In [3]:
HORIZON_MAP = {
    1: 'short',
    3: "medium",
    10: "long_short",
    25: "long_long"
}

# 23 entity unique codes (code)
# 47 entity segments (sub_code)
# 5 parent categorical labels describing where the entity belongs (sub_category)
SUB_CAT = [
    "PZ9S1Z4V"
    "V8BKY1IV",
    "PHHHVYZI",
    "NQ58FVQM",
    "DPPUO5X2"
]

cols = ['feature_a','feature_b','feature_c','feature_d','feature_e','feature_f','feature_g','feature_h','feature_i','feature_j','feature_k',
 'feature_l','feature_m','feature_n','feature_o','feature_p','feature_q','feature_r','feature_s','feature_t','feature_u','feature_v','feature_w',
 'feature_x','feature_y','feature_z','feature_aa','feature_ab','feature_ac','feature_ad','feature_ae','feature_af','feature_ag','feature_ah','feature_ai',
 'feature_aj','feature_ak','feature_al','feature_am','feature_an','feature_ao','feature_ap','feature_aq','feature_ar','feature_as','feature_at',
 'feature_au','feature_av','feature_aw','feature_ax','feature_ay','feature_az','feature_ba','feature_bb','feature_bc','feature_bd','feature_be',
 'feature_bf','feature_bg','feature_bh','feature_bi','feature_bj','feature_bk','feature_bl','feature_bm','feature_bn','feature_bo','feature_bp',
 'feature_bq','feature_br','feature_bs','feature_bt','feature_bu','feature_bv','feature_bw','feature_bx','feature_by','feature_bz','feature_ca',
 'feature_cb','feature_cc','feature_cd','feature_ce','feature_cf','feature_cg','feature_ch']

class CollarTag:
    """
        Example usage:

            ```python
            # (Encode Columns - Optional)

            df.with_columns(
                [
                    pl.col('sub_category')
                    .replace(SubCat.tag2id)
                    .cast(pl.Int32),
                    pl.col('sub_code')
                    .replace(SubCodeCat.tag2id)
                    .cast(pl.Int32),
                    pl.col('code')
                    .replace(CodeCat.tag2id)
                    .cast(pl.Int32)
                ]
            )
            ```
    """
    def __init__(self, inputs: list[str] | dict):

        if isinstance(inputs, dict): 
            print('Received dict type to encode. Encoding based on the assumption that it is HORIZON_MAP. If not, pls update this code.')
            self.labels = sorted(list(inputs.values()), reverse=False)
            self.id2tag = inputs
            self.tag2id = {tag: idx for idx, tag in inputs.items()}
        else:
            if not isinstance(inputs, list):
                print(f'Expected inputs to be list but received {type(inputs)}. Converting to list now')
                inputs = list(inputs)
                
            self.labels = sorted(inputs, reverse=False)
            self.id2tag: dict = {idx: tag for idx, tag in enumerate(self.labels)}
            self.tag2id: dict = {tag: idx for idx, tag in enumerate(self.labels)}

#CodeCat = CollarTag(df['code'].unique())
#SubCodeCat = CollarTag(df['sub_code'].unique())
#SubCat = CollarTag(df['sub_category'].unique())
#HorizonCat = CollarTag(HORIZON_MAP)

In [ ]:
# Load model
df = pl.read_parquet(TRAIN_PATH, n_rows=SAMPLE_SIZE)
# Data Preprocessing Stage I
# To prepare the features for training and avoid future values 
group_cols=['code', 'sub_code', 'sub_category']

df = df.sort(['code', 'sub_code', 'sub_category', 'ts_index'])

# Transform / Encode Temporal Features: ['horizon', 'ts_index']
df = df.with_columns(
    pl.col('ts_index')
    .rank("dense")
    .over(group_cols)
    .alias("ts_rank")
).with_columns(
    (pl.col('ts_rank') / pl.col('ts_rank').max().over(group_cols))
    .alias('ts_rank_norm')
).with_columns([
    (pl.col('ts_rank_norm') * 2 * np.pi).sin().alias('ts_sin'),
    (pl.col('ts_rank_norm') * 2 * np.pi).cos().alias('ts_cos'),
    (pl.col('ts_rank_norm').cut([0.3, 0.5, 0.7]).alias('ts_phase'))
])

TsPhase = CollarTag(df['ts_phase'].unique())

df = df.with_columns(
    pl.col('ts_phase')
    .cast(pl.String)
    .replace(TsPhase.tag2id)
    .cast(pl.Int32)
)

assert set(['ts_index', 'ts_rank', 'ts_rank_norm', 'ts_sin', 'ts_cos', 'ts_phase']).issubset(set(df.columns))


Expected inputs to be list but received <class 'polars.series.series.Series'>. Converting to list now


In [63]:
# Additional Features 

lags = list(range(1, len(TsPhase.labels) + 1))  # [1, 2, 3, 4]

print(f'Determining Lags (Prior Lookback Windows) for {lags}')

roll_mean_cols = [
    f"{col}_roll_mean_{lag}"
    for col in cols 
    for lag in lags
]

roll_std_cols = [
    f"{col}_roll_std_{lag}"
    for col in cols 
    for lag in lags
]

momentum_cols = [
    f"{col}_diff_{lag}"
    for col in cols 
    for lag in lags
]

df = df.with_columns([
    pl.col(col).shift(lag).rolling_std(window_size=lag)
      .over(group_cols).alias(f"{col}_roll_std_{lag}")
    for col in cols for lag in lags
]).with_columns([
    pl.col(col).shift(lag).rolling_mean(window_size=lag)
      .over(group_cols).alias(f"{col}_roll_mean_{lag}")
    for col in cols for lag in lags
]).with_columns([
    (pl.col(col) - pl.col(col).shift(lag))
      .over(group_cols)                        # removed horizon here
      .alias(f"{col}_diff_{lag}")
    for col in cols for lag in lags
])

feature_cols = cols + roll_mean_cols + roll_std_cols + momentum_cols

assert set(feature_cols).issubset(set(df.columns))

df = df.with_columns(
    pl.col(pl.NUMERIC_DTYPES).fill_nan(None)
)

Determining Lags (Prior Lookback Windows) for [1, 2, 3, 4]


/tmp/ipykernel_880/4268000295.py:45: DeprecationWarning: `NUMERIC_DTYPES` is deprecated. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  pl.col(pl.NUMERIC_DTYPES).fill_nan(None)


In [48]:
(
    df
    .filter(pl.col("horizon") == 10)
    .select([
        pl.corr(pl.col(c), pl.col("y_target")).abs().alias(c)
        for c in feature_cols
    ])
    .transpose(include_header=True)
    .rename({"column": "feature", "column_0": "abs_corr"})
    .drop_nulls()
    .drop_nans()
    .sort("abs_corr", descending=True)
    .head(10)
)

feature,abs_corr
str,f64
"""feature_cb""",0.221333
"""feature_cd_roll_mean_2""",0.221314
"""feature_cd""",0.219153
"""feature_cb_roll_mean_1""",0.218944
"""feature_cd_roll_mean_1""",0.218939
"""feature_cb_roll_mean_2""",0.217212
"""feature_cd_roll_mean_3""",0.216616
"""feature_cb_roll_mean_3""",0.214723
"""feature_cc""",0.212093


In [87]:
# predict_short_horizon
# predict_medium_horizon
# predict_long_short_horizon
# predict_long_horizon
"""
df.filter(
    (pl.col('code') == "W2MW3G2L") &
    (pl.col('sub_code') == "A9HVWZIA") &
    (pl.col('sub_category') == "PZ9S1Z4V") &
    (pl.col('horizon') == 1) 
)['ts_index'].unique()

"""

'\ndf.filter(\n    (pl.col(\'code\') == "W2MW3G2L") &\n    (pl.col(\'sub_code\') == "A9HVWZIA") &\n    (pl.col(\'sub_category\') == "PZ9S1Z4V") &\n    (pl.col(\'horizon\') == 1) \n)[\'ts_index\'].unique()\n\n'

# Train Model

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

# --- Column definitions ---
meta_cols = ["code", "sub_code", "sub_category"]

# --- Base preprocessor (cloned per horizon) ---
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), meta_cols),
        ("num", "passthrough", feature_cols),
    ]
).set_output(transform="pandas")  

selector_estimator = LGBMRegressor(
    n_estimators=200,       # lightweight — just for importance ranking
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

selector = SelectFromModel(
    estimator=selector_estimator,
    threshold="mean",       # keep features with importance >= mean importance
)

lgbm_params = dict(
    n_estimators=1000,
    learning_rate=0.03,       
    max_depth=4,              
    num_leaves=31,          
    subsample=0.6,           
    colsample_bytree=0.6,     
    min_child_samples=50,    
    reg_alpha=0.5,            
    reg_lambda=1.0,           
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
# --- Train one model per horizon ---
models = {}
partitions = df_train.partition_by("horizon", as_dict=True)

for (h,), horizon_df in partitions.items():
    train = horizon_df.filter(pl.col("ts_phase").is_in([0, 1, 2]))
    test  = horizon_df.filter(pl.col("ts_phase") == 3)

    X_train = train.select(meta_cols + feature_cols).to_pandas()
    X_test  = test.select(meta_cols + feature_cols).to_pandas()
    y_train = train["y_target"].to_pandas()
    y_test  = test["y_target"].to_pandas()
    w_train = train["weight"].to_pandas()
    w_test  = test["weight"].to_pandas()

    # Preprocess the dataset
    preprocessor_h = clone(preprocessor)
    X_train_t = preprocessor_h.fit_transform(X_train)
    X_test_t  = preprocessor_h.transform(X_test)

    # Select best features
    selector_h = clone(selector)
    selector_h.fit(X_train_t, y_train)
    X_train_s = selector_h.transform(X_train_t)
    X_test_s  = selector_h.transform(X_test_t)

    n_total    = X_train_t.shape[1]
    n_selected = X_train_s.shape[1]
    print(f"Horizon {h} | Features selected: {n_selected}/{n_total}")

    model = LGBMRegressor(**lgbm_params)
    model.fit(
        X_train_s, y_train,
        sample_weight=w_train,
        eval_set=[(X_test_s, y_test)],
        eval_sample_weight=[w_test],
        callbacks=[early_stopping(50, verbose=False), log_evaluation(period=-1)],
    )
    
    models[f"h{h}"] = (preprocessor_h, selector_h, model)

    train_preds = model.predict(X_train_s) 
    test_preds  = model.predict(X_test_s)  

    # Convert to numpy to ensure consistent types throughout eval
    y_train_np = y_train.to_numpy()
    y_test_np  = y_test.to_numpy()
    w_train_np = w_train.to_numpy()
    w_test_np  = w_test.to_numpy()

    # Correct arg order: eval(y_target, y_pred, w)
    print(f"Horizon {h} | TRAIN:", eval(y_train_np, train_preds, w_train_np))
    print(f"Horizon {h} | TEST: ", eval(y_test_np,  test_preds,  w_test_np))

Horizon 10 | Features selected: 312/1121


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Horizon 10 | TRAIN: {'weighted_rmse_score': 0.7700806658779126, 'residue': array([-0.47082812, -0.42200261, -0.49196063, ..., -0.44471662,
       -0.45121883, -0.09489152]), 'rmse': 0.217, 'bias^2': 0.0, 'var': 0.047, 'wrmse_rmse': 0.1625}
Horizon 10 | TEST:  {'weighted_rmse_score': 0.420831747545941, 'residue': array([-0.29144449, -0.20103568, -0.2375389 , ...,  0.1106611 ,
       -0.02564304,  0.10354474]), 'rmse': 0.2989, 'bias^2': 0.0006, 'var': 0.0887, 'wrmse_rmse': 0.2439}
Horizon 25 | Features selected: 300/1121


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Horizon 25 | TRAIN: {'weighted_rmse_score': 0.9201924875259072, 'residue': array([ 0.07854149, -0.090941  , -0.0499192 , ...,  0.04434308,
       -0.23009548, -0.14978542]), 'rmse': 0.2264, 'bias^2': 0.0001, 'var': 0.0512, 'wrmse_rmse': 0.1454}
Horizon 25 | TEST:  {'weighted_rmse_score': 0.5761970719989399, 'residue': array([-0.20324751, -0.44973681, -0.4887281 , ..., -0.16853398,
       -0.09396864, -0.0876019 ]), 'rmse': 0.375, 'bias^2': 0.0008, 'var': 0.1398, 'wrmse_rmse': 0.2973}
Horizon 1 | Features selected: 365/1121


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Horizon 1 | TRAIN: {'weighted_rmse_score': 0.53106669923846, 'residue': array([ 0.07392026, -0.05700788,  0.1346384 , ..., -0.00860033,
       -0.04572063, -0.05171311]), 'rmse': 0.1006, 'bias^2': 0.0, 'var': 0.0101, 'wrmse_rmse': 0.0732}
Horizon 1 | TEST:  {'weighted_rmse_score': 0.199345487270873, 'residue': array([-0.04618675, -0.05829833, -0.20135055, ...,  0.09442578,
       -0.00807891, -0.29982769]), 'rmse': 0.1237, 'bias^2': 0.0, 'var': 0.0153, 'wrmse_rmse': 0.0984}
Horizon 3 | Features selected: 330/1121


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Horizon 3 | TRAIN: {'weighted_rmse_score': 0.74577610537503, 'residue': array([ 0.09681289, -0.20558451, -0.12584613, ..., -0.10926006,
       -0.1924214 , -0.33842678]), 'rmse': 0.139, 'bias^2': 0.0, 'var': 0.0193, 'wrmse_rmse': 0.1007}
Horizon 3 | TEST:  {'weighted_rmse_score': 0.3790460217717106, 'residue': array([-0.32598531,  0.00433968,  0.0911268 , ..., -0.0090728 ,
        0.12316049,  0.15013387]), 'rmse': 0.1958, 'bias^2': 0.0, 'var': 0.0383, 'wrmse_rmse': 0.1575}


----

# REfactorsed

In [ ]:
group_cols=['code', 'sub_code', 'sub_category']

def add_temporal_features(
    df: pl.DataFrame,
    group_cols: list[str] = ['code', 'sub_code', 'sub_category'],
    phase_cuts: list[float] = [0.3, 0.5, 0.7],
    ts_phase_tag: CollarTag | None = None,
) -> tuple[pl.DataFrame, CollarTag]:
    """Adds ts_rank, ts_rank_norm, ts_sin, ts_cos, ts_phase to the dataframe."""

    df = df.sort(group_cols + ['ts_index'])

    df = df.with_columns(
        pl.col('ts_index').rank("dense").over(group_cols).alias("ts_rank")
    ).with_columns(
        (pl.col('ts_rank') / pl.col('ts_rank').max().over(group_cols)).alias('ts_rank_norm')
    ).with_columns([
        (pl.col('ts_rank_norm') * 2 * np.pi).sin().alias('ts_sin'),
        (pl.col('ts_rank_norm') * 2 * np.pi).cos().alias('ts_cos'),
        pl.col('ts_rank_norm').cut(phase_cuts).alias('ts_phase'),
    ])

    if ts_phase_tag is None:
        ts_phase_tag = CollarTag(df['ts_phase'].unique())

    df = df.with_columns(
        pl.col('ts_phase').cast(pl.String).replace(ts_phase_tag.tag2id).cast(pl.Int32)
    )

    assert set(['ts_index', 'ts_rank', 'ts_rank_norm', 'ts_sin', 'ts_cos', 'ts_phase']).issubset(set(df.columns))

    return df, ts_phase_tag


def add_lag_features(
    df: pl.DataFrame,
    cols: list[str],
    lags: list[int],
    group_cols: list[str] = ['code', 'sub_code', 'sub_category'],
) -> tuple[pl.DataFrame, list[str], list[str], list[str]]:
    """Adds rolling mean, rolling std, and momentum diff lag features."""

    roll_mean_cols = [f"{col}_roll_mean_{lag}" for col in cols for lag in lags]
    roll_std_cols  = [f"{col}_roll_std_{lag}"  for col in cols for lag in lags]
    momentum_cols  = [f"{col}_diff_{lag}"      for col in cols for lag in lags]

    df = df.with_columns([
        pl.col(col).shift(lag).rolling_std(window_size=lag)
          .over(group_cols).alias(f"{col}_roll_std_{lag}")
        for col in cols for lag in lags
    ]).with_columns([
        pl.col(col).shift(lag).rolling_mean(window_size=lag)
          .over(group_cols).alias(f"{col}_roll_mean_{lag}")
        for col in cols for lag in lags
    ]).with_columns([
        (pl.col(col) - pl.col(col).shift(lag))
          .over(group_cols).alias(f"{col}_diff_{lag}")
        for col in cols for lag in lags
    ])

    assert set(cols + roll_mean_cols + roll_std_cols + momentum_cols).issubset(set(df.columns))

    return df, roll_mean_cols, roll_std_cols, momentum_cols


def preprocess(
    df: pl.DataFrame,
    cols: list[str],
    group_cols: list[str] = ['code', 'sub_code', 'sub_category'],
    phase_cuts: list[float] = [0.3, 0.5, 0.7],
    ts_phase_tag: CollarTag | None = None,  # pass fitted tag for test set
) -> tuple[pl.DataFrame, list[str], CollarTag]:
    """
    Full preprocessing pipeline.

    For train : call with ts_phase_tag=None  → fits and returns a new CollarTag
    For test  : call with ts_phase_tag=<fitted tag from train> → reuses train encoding
    """

    df, ts_phase_tag = add_temporal_features(df, group_cols, phase_cuts, ts_phase_tag=ts_phase_tag if ts_phase_tag else None)
    print('Finished adding temporal features')

    lags = list(range(1, len(ts_phase_tag.labels) + 1))
    print(f'Determining Lags (Prior Lookback Windows): {lags}')

    df, roll_mean_cols, roll_std_cols, momentum_cols = add_lag_features(df, cols, lags, group_cols)

    feature_cols = cols + roll_mean_cols + roll_std_cols + momentum_cols

    df = (
            df
            .with_columns(pl.col(pl.NUMERIC_DTYPES).fill_nan(None))
            .with_columns(pl.col(feature_cols).fill_null(0.0))
    )
    print("Finished resolving nulls/nans")
    
    return df, feature_cols, ts_phase_tag

ds_train = pl.read_parquet(TRAIN_PATH, n_rows=SAMPLE_SIZE)
ds_test = pl.read_parquet(TEST_PATH)
df_train, feature_cols, ts_phase_tag = preprocess(ds_train, cols=cols)
df_test, _, _ = preprocess(ds_test, cols=cols, ts_phase_tag=ts_phase_tag)

Expected inputs to be list but received <class 'polars.series.series.Series'>. Converting to list now
Finished adding temporal features
Determining Lags (Prior Lookback Windows): [1, 2, 3, 4]


/tmp/ipykernel_1204/4291170577.py:92: DeprecationWarning: `NUMERIC_DTYPES` is deprecated. Define your own data type groups or use the `polars.selectors` module for selecting columns of a certain data type.
  .with_columns(pl.col(pl.NUMERIC_DTYPES).fill_nan(None))


Finished resolving nulls/nans
Finished adding temporal features
Determining Lags (Prior Lookback Windows): [1, 2, 3, 4]
Finished resolving nulls/nans


In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from lightgbm import LGBMRegressor, early_stopping, log_evaluation

# --- Column definitions ---
meta_cols = ["code", "sub_code", "sub_category"]

# --- Base preprocessor (cloned per horizon) ---
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), meta_cols),
        ("num", "passthrough", feature_cols),
    ]
).set_output(transform="pandas")  

selector_estimator = LGBMRegressor(
    n_estimators=200,       # lightweight — just for importance ranking
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

selector = SelectFromModel(
    estimator=selector_estimator,
    threshold="mean",       # keep features with importance >= mean importance
)

lgbm_params = dict(
    n_estimators=1000,
    learning_rate=0.03,       
    max_depth=4,              
    num_leaves=31,          
    subsample=0.6,           
    colsample_bytree=0.6,     
    min_child_samples=50,    
    reg_alpha=0.5,            
    reg_lambda=1.0,           
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
# --- Train one model per horizon ---
models = {}
partitions = df_train.partition_by("horizon", as_dict=True)

for (h,), horizon_df in partitions.items():
    train = horizon_df.filter(pl.col("ts_phase").is_in([0, 1, 2]))
    eval  = horizon_df.filter(pl.col("ts_phase") == 3)

    X_train = train.select(meta_cols + feature_cols).to_pandas()
    X_eval  = eval.select(meta_cols + feature_cols).to_pandas()
    y_train = train["y_target"].to_pandas()
    y_eval  = eval["y_target"].to_pandas()
    w_train = train["weight"].to_pandas()
    w_eval  = eval["weight"].to_pandas()

    # Preprocess the dataset
    preprocessor_h = clone(preprocessor)
    X_train_t = preprocessor_h.fit_transform(X_train)
    X_eval_t  = preprocessor_h.transform(X_eval)

    # Select best features
    selector_h = clone(selector)
    selector_h.fit(X_train_t, y_train)
    X_train_s = selector_h.transform(X_train_t)
    X_eval_s  = selector_h.transform(X_eval_t)

    n_total    = X_train_t.shape[1]
    n_selected = X_train_s.shape[1]
    print(f"Horizon {h} | Features selected: {n_selected}/{n_total}")

    model = LGBMRegressor(**lgbm_params)
    model.fit(
        X_train_s, y_train,
        sample_weight=w_train,
        eval_set=[(X_eval_s, y_eval)],
        eval_sample_weight=[w_eval],
        callbacks=[early_stopping(50, verbose=False), log_evaluation(period=-1)],
    )
    
    models[f"h{h}"] = (preprocessor_h, selector_h, model)

    train_preds = model.predict(X_train_s) 
    eval_preds  = model.predict(X_eval_s)  

    # Convert to numpy to ensure consistent types throughout eval
    y_train_np = y_train.to_numpy()
    y_eval_np  = y_eval.to_numpy()
    w_train_np = w_train.to_numpy()
    w_eval_np  = w_eval.to_numpy()

    # Correct arg order: eval(y_target, y_pred, w)
    print(f"Horizon {h} | TRAIN:", eval(y_train_np, train_preds, w_train_np))
    print(f"Horizon {h} | EVAL: ", eval(y_eval_np,  eval_preds,  w_eval_np))

Horizon 10 | Features selected: 307/1121


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Horizon 10 | TRAIN: {'weighted_rmse_score': 0.7363867877699333, 'residue': array([-0.52166399, -0.53679535, -0.53682717, ..., -0.57405231,
       -0.48143954, -0.17428051]), 'rmse': 0.2299, 'bias^2': 0.0001, 'var': 0.0528, 'wrmse_rmse': 0.1723}
Horizon 10 | TEST:  {'weighted_rmse_score': 0.3942718933685792, 'residue': array([-0.26756638, -0.23771944, -0.2829018 , ...,  0.06523156,
        0.04451963,  0.1173084 ]), 'rmse': 0.301, 'bias^2': 0.0008, 'var': 0.0898, 'wrmse_rmse': 0.247}
Horizon 25 | Features selected: 291/1121


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Horizon 25 | TRAIN: {'weighted_rmse_score': 0.8713196251267166, 'residue': array([ 0.04249461, -0.11936171, -0.10819435, ..., -0.02362738,
       -0.22685958, -0.25984197]), 'rmse': 0.2762, 'bias^2': 0.0001, 'var': 0.0762, 'wrmse_rmse': 0.1822}
Horizon 25 | TEST:  {'weighted_rmse_score': 0.510242863213803, 'residue': array([-0.2709525 , -0.36140461, -0.41665882, ..., -0.17345233,
       -0.12085726, -0.12321913]), 'rmse': 0.3956, 'bias^2': 0.0024, 'var': 0.1541, 'wrmse_rmse': 0.3128}
Horizon 1 | Features selected: 355/1121


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Horizon 1 | TRAIN: {'weighted_rmse_score': 0.5750608512917397, 'residue': array([ 0.07524508, -0.05115056,  0.13081447, ...,  0.00407259,
       -0.03011392, -0.06545646]), 'rmse': 0.0982, 'bias^2': 0.0, 'var': 0.0096, 'wrmse_rmse': 0.0706}
Horizon 1 | TEST:  {'weighted_rmse_score': 0.2181445507798921, 'residue': array([-0.04102641, -0.06277165, -0.20330838, ...,  0.06735844,
       -0.02151698, -0.30536582]), 'rmse': 0.1232, 'bias^2': 0.0, 'var': 0.0152, 'wrmse_rmse': 0.098}
Horizon 3 | Features selected: 335/1121


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Horizon 3 | TRAIN: {'weighted_rmse_score': 0.8233812139387006, 'residue': array([ 0.09486701, -0.1288257 , -0.08899551, ..., -0.04113033,
       -0.16876349, -0.29168197]), 'rmse': 0.1221, 'bias^2': 0.0, 'var': 0.0149, 'wrmse_rmse': 0.0858}
Horizon 3 | TEST:  {'weighted_rmse_score': 0.38503291460762723, 'residue': array([-0.27687402,  0.03944522,  0.13641849, ...,  0.03012499,
        0.07328417,  0.07844244]), 'rmse': 0.1943, 'bias^2': 0.0002, 'var': 0.0376, 'wrmse_rmse': 0.1571}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [8]:
df = pl.read_parquet(TEST_PATH)
df.head()

id,code,sub_code,sub_category,horizon,ts_index,feature_a,feature_b,feature_c,feature_d,feature_e,feature_f,feature_g,feature_h,feature_i,feature_j,feature_k,feature_l,feature_m,feature_n,feature_o,feature_p,feature_q,feature_r,feature_s,feature_t,feature_u,feature_v,feature_w,feature_x,feature_y,feature_z,feature_aa,feature_ab,feature_ac,feature_ad,feature_ae,…,feature_ax,feature_ay,feature_az,feature_ba,feature_bb,feature_bc,feature_bd,feature_be,feature_bf,feature_bg,feature_bh,feature_bi,feature_bj,feature_bk,feature_bl,feature_bm,feature_bn,feature_bo,feature_bp,feature_bq,feature_br,feature_bs,feature_bt,feature_bu,feature_bv,feature_bw,feature_bx,feature_by,feature_bz,feature_ca,feature_cb,feature_cc,feature_cd,feature_ce,feature_cf,feature_cg,feature_ch
str,str,str,str,i32,i32,i32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…","""W2MW3G2L""","""495MGHFJ""","""PZ9S1Z4V""",3,3647,95,10.365266,3.209321,8.109339,9.043471,10.123041,15.722121,0.000243,0.021819,0.00142,0.000073,0.572125,1.265875,1.341192,0.005564,0.011987,0.035243,0.833918,1.791284,0.020539,0.218876,0.08066,-50.98124,-4.854592,-8.087713,119.237254,0.040442,0.635006,0.105355,0.075415,0.03444,…,197.344987,209.253182,0.016366,0.552138,108.859861,2.369993,66589.814887,34282.221003,1316.738008,0.04801,11660.961097,0.116372,11.122246,716.158132,0.008559,1.772256,38.452077,0.872948,0.06611,0.078856,0.030888,-0.480743,-0.197747,-3.659776,0.100295,3.131395,4.554259,-0.000832,-0.032241,-0.00083,-0.058961,-0.002774,-0.00148,-0.25646,1.665532,0.071324,2
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…","""W2MW3G2L""","""495MGHFJ""","""PZ9S1Z4V""",10,3647,88,2.571477,15.234848,16.505699,0.230426,10.145378,10.159641,0.000243,0.021819,0.00142,0.000073,0.572125,1.265875,1.341192,0.005564,0.011987,0.035243,0.833918,1.791284,0.020539,0.218876,0.08066,-50.98124,-4.854592,-8.087713,119.237254,0.040442,0.635006,0.105355,0.075415,0.03444,…,197.344987,209.253182,0.016366,0.552138,108.859861,2.369993,66589.814887,34282.221003,1316.738008,0.04801,11660.961097,0.116372,11.122246,716.158132,0.008559,1.772256,38.452077,0.872948,0.06611,0.078856,0.030888,-0.480743,-0.197747,-3.659776,0.100295,3.131395,4.554259,-0.000832,-0.032241,-0.00083,-0.058961,-0.002774,-0.00148,-0.25646,1.665532,0.071324,2
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…","""W2MW3G2L""","""495MGHFJ""","""PZ9S1Z4V""",25,3647,71,5.524709,6.931663,8.939537,0.668187,16.578701,3.15069,0.000243,0.021819,0.00142,0.000073,0.572125,1.265875,1.341192,0.005564,0.011987,0.035243,0.833918,1.791284,0.020539,0.218876,0.08066,-50.98124,-4.854592,-8.087713,119.237254,0.040442,0.635006,0.105355,0.075415,0.03444,…,197.344987,209.253182,0.016366,0.552138,108.859861,2.369993,66589.814887,34282.221003,1316.738008,0.04801,11660.961097,0.116372,11.122246,716.158132,0.008559,1.772256,38.452077,0.872948,0.06611,0.078856,0.030888,-0.480743,-0.197747,-3.659776,0.100295,3.131395,4.554259,-0.000832,-0.032241,-0.00083,-0.058961,-0.002774,-0.00148,-0.25646,1.665532,0.071324,2
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…","""W2MW3G2L""","""495MGHFJ""","""PZ9S1Z4V""",1,3647,97,10.293758,14.89366,9.435544,2.335377,3.477961,15.680595,0.000243,0.021819,0.00142,0.000073,0.572125,1.265875,1.341192,0.005564,0.011987,0.035243,0.833918,1.791284,0.020539,0.218876,0.08066,-50.98124,-4.854592,-8.087713,119.237254,0.040442,0.635006,0.105355,0.075415,0.03444,…,197.344987,209.253182,0.016366,0.552138,108.859861,2.369993,66589.814887,34282.221003,1316.738008,0.04801,11660.961097,0.116372,11.122246,716.158132,0.008559,1.772256,38.452077,0.872948,0.06611,0.078856,0.030888,-0.480743,-0.197747,-3.659776,0.100295,3.131395,4.554259,-0.000832,-0.032241,-0.00083,-0.058961,-0.002774,-0.00148,-0.25646,1.665532,0.071324,2
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…",

In [ ]:
X = df.select(cols ).fill_null(0).to_numpy()
df = df.with_columns(pl.Series(name="prediction", values=model.predict(X)))
df.select(['id','prediction']).head()
df.select(['id','prediction']).write_csv('submission.csv')

id,prediction
str,f64
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…",-0.013161
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…",-0.013091
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…",-0.013138
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…",-0.013245
"""W2MW3G2L__495MGHFJ__PZ9S1Z4V__…",-0.013216


In [10]:
df.select(['id','prediction']).write_csv('submission.csv')

In [ ]:
def add_lag_features(
    df: pl.DataFrame,
    cols: list[str],
    lags: list[int],
    group_cols: list[str] = ['code', 'sub_code', 'sub_category'],
) -> tuple[pl.DataFrame, list[str], list[str], list[str]]:
    """Adds rolling mean, rolling std, and momentum diff lag features."""

    roll_mean_cols = [f"{col}_roll_mean_{lag}" for col in cols for lag in lags]
    roll_std_cols  = [f"{col}_roll_std_{lag}"  for col in cols for lag in lags]
    momentum_cols  = [f"{col}_diff_{lag}"      for col in cols for lag in lags]

    df = df.with_columns([
        pl.col(col).shift(lag).rolling_std(window_size=lag)
          .over(group_cols).alias(f"{col}_roll_std_{lag}")
        for col in cols for lag in lags
    ]).with_columns([
        pl.col(col).shift(lag).rolling_mean(window_size=lag)
          .over(group_cols).alias(f"{col}_roll_mean_{lag}")
        for col in cols for lag in lags
    ]).with_columns([
        (pl.col(col) - pl.col(col).shift(lag))
          .over(group_cols).alias(f"{col}_diff_{lag}")
        for col in cols for lag in lags
    ])

    assert set(cols + roll_mean_cols + roll_std_cols + momentum_cols).issubset(set(df.columns))

    return df, roll_mean_cols, roll_std_cols, momentum_cols



In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectFromModel
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
import warnings

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module="sklearn.utils.validation"
)

SEED = 42

# --- Base preprocessor (cloned per horizon) ---
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), group_cols),
        ("num", "passthrough", feature_cols),
    ]
).set_output(transform="pandas")  

selector_estimator = LGBMRegressor(
    n_estimators       = 500,        # ↑ from 200 — more stable importances at 5M rows
    n_jobs             = -1,
    verbose            = -1,
)

selector = SelectFromModel(
    estimator=selector_estimator,
    threshold="mean"
)

lgbm_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'feature_fraction': 0.9,
    'random_state': SEED,
    'n_estimators': 2000,
    'n_jobs': -1,
    'verbose': -1,
    'learning_rate': 0.06471921325020952,
    'num_leaves': 296,
    'max_depth': 12,
    'min_child_samples': 88,
    'subsample': 0.7891691017340575,
    'colsample_bytree': 0.9037484715991748,
    'reg_alpha': 1.407892012906228,
    'reg_lambda': 7.212991559021246
}

# Train one model per horizon
models = {}

partitions = df_train.partition_by("horizon", as_dict=True)

for (h,), horizon_df in partitions.items():
    
    l, k = None, None 
    if h == 1:
        l = [0]
        k = 0
    elif h == 3: 
        l = [0]
        k = 1
    elif h == 10:
        l = [0, 1]
        k = 2
    elif h == 25:
        l = [0, 1, 2]
        k = 3
        
    train = horizon_df.filter(pl.col("ts_phase").is_in(l))
    eval  = horizon_df.filter(pl.col("ts_phase") == k)

    X_train = train.select(meta_cols + feature_cols).to_pandas()
    X_eval  = eval.select(meta_cols + feature_cols).to_pandas()
    y_train = train["y_target"].to_pandas()
    y_eval  = eval["y_target"].to_pandas()
    w_train = train["weight"].to_pandas()
    w_eval  = eval["weight"].to_pandas()

    # Preprocess the dataset
    preprocessor_h = clone(preprocessor)
    X_train_t = preprocessor_h.fit_transform(X_train)
    X_eval_t  = preprocessor_h.transform(X_eval)

    # Select best features
    selector_h = clone(selector)
    selector_h.fit(X_train_t, y_train)
    X_train_s = selector_h.transform(X_train_t)
    X_eval_s  = selector_h.transform(X_eval_t)

    n_total    = X_train_t.shape[1]
    n_selected = X_train_s.shape[1]
    print(f"Horizon {h} | Features selected: {n_selected}/{n_total}")

    model = LGBMRegressor(**lgbm_params)
    model.fit(
        X_train_s, y_train,
        sample_weight=w_train,
        eval_set=[(X_eval_s, y_eval)],
        eval_sample_weight=[w_eval],
        callbacks=[
            early_stopping(50, verbose=False),
            log_evaluation(50)
        ],
    )
    
    models[f"h{h}"] = (preprocessor_h, selector_h, model)

    train_preds = model.predict(X_train_s) 
    eval_preds  = model.predict(X_eval_s)  

    # Convert to numpy to ensure consistent types throughout eval
    y_train_np = y_train.to_numpy()
    y_eval_np  = y_eval.to_numpy()
    w_train_np = w_train.to_numpy()
    w_eval_np  = w_eval.to_numpy()

    print(f"Horizon {h} | TRAIN:", evaluate(y_train_np, train_preds, w_train_np))
    print(f"Horizon {h} | EVAL: ", evaluate(y_eval_np,  eval_preds,  w_eval_np))

Horizon 10 | Features selected: 32/89
[50]	valid_0's rmse: 0.170894
[100]	valid_0's rmse: 0.171494
Horizon 10 | TRAIN: {'weighted_rmse_score': 0.807468233668456, 'rmse': 0.1929, 'bias^2': 0.0, 'var': 0.0372, 'wrmse_rmse': 0.1224}
Horizon 10 | EVAL:  {'weighted_rmse_score': 0.5225612894810172, 'rmse': 0.2259, 'bias^2': 0.0006, 'var': 0.0505, 'wrmse_rmse': 0.1705}
Horizon 25 | Features selected: 34/89
[50]	valid_0's rmse: 0.255116
[100]	valid_0's rmse: 0.253286
Horizon 25 | TRAIN: {'weighted_rmse_score': 0.926195654993121, 'rmse': 0.1981, 'bias^2': 0.0, 'var': 0.0392, 'wrmse_rmse': 0.1152}
Horizon 25 | EVAL:  {'weighted_rmse_score': 0.5452079789401277, 'rmse': 0.3399, 'bias^2': 0.0004, 'var': 0.1152, 'wrmse_rmse': 0.2532}
Horizon 1 | Features selected: 28/89
[50]	valid_0's rmse: 0.0683922
[100]	valid_0's rmse: 0.0625654
[150]	valid_0's rmse: 0.0578464
[200]	valid_0's rmse: 0.0543327
[250]	valid_0's rmse: 0.0510305
[300]	valid_0's rmse: 0.0484411
[350]	valid_0's rmse: 0.0457568
[400]	vali